# Paper main result: cumulative time, latency, and recall

This notebook reads the released `results/drip/acc_time_curves.csv` table, then replaces DRIP/HNSW series with the latest complete matching raw runs in the same directory. Run `gzip -dk results/drip/*.csv.gz` once after cloning.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def locate_repo_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'results' / 'drip').is_dir():
            return candidate
    raise RuntimeError('Could not locate results/drip from the current working directory')


REPO_ROOT = locate_repo_root()
RESULTS_DIR = REPO_ROOT / 'results'
if str(RESULTS_DIR) not in sys.path:
    sys.path.insert(0, str(RESULTS_DIR))
from latest_results import load_acc_time_curves

DATA_DIR = RESULTS_DIR / 'drip'
FIGURE_DIR = REPO_ROOT / 'results' / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')

DATASET_ORDER = ['MSMARCO', 'SIFT', 'Laion-I2I', 'Wiki']
METHOD_ORDER = ['DRIP', 'CrackIVF$^*$', 'HNSW', 'DiskANN', 'IVF4096', 'Flat']
METHOD_COLORS = {
    'DRIP': '#ec003d',
    'CrackIVF$^*$': '#008a55',
    'HNSW': '#2666e6',
    'DiskANN': '#d89000',
    'IVF4096': '#5e0fa8',
    'Flat': '#999999',
}


In [ ]:
curves = load_acc_time_curves(DATA_DIR)

fig, axes = plt.subplots(
    3,
    len(DATASET_ORDER),
    figsize=(18, 9),
    gridspec_kw={'height_ratios': [1, 1, 0.65]},
)

legend_handles = {}
for column, dataset in enumerate(DATASET_ORDER):
    dataset_curves = curves[curves['dataset'] == dataset]
    axes[0, column].set_title(dataset, fontsize=16, pad=10)

    for method in METHOD_ORDER:
        method_curves = dataset_curves[dataset_curves['method'] == method].sort_values('num_queries')
        if method_curves.empty:
            continue

        linestyle = method_curves['linestyle'].iloc[0]
        color = METHOD_COLORS[method]
        x = method_curves['num_queries']

        line = axes[0, column].plot(
            x,
            method_curves['cumulative_wall_time_s'],
            color=color,
            linestyle=linestyle,
            linewidth=1.8,
            alpha=0.75,
            label=method,
        )[0]
        legend_handles.setdefault(method, line)

        axes[1, column].plot(
            x,
            method_curves['query_latency_ms'],
            color=color,
            linestyle=linestyle,
            linewidth=1.8,
            alpha=0.75,
        )
        axes[2, column].plot(
            x,
            method_curves['recall'],
            color=color,
            linestyle=':' if 'CrackIVF' in method else linestyle,
            linewidth=1.8,
            alpha=0.75,
        )

    loaded_max_queries = dataset_curves['num_queries'].max()
    max_queries = 10 ** round(np.log10(loaded_max_queries))
    ticks = [10 ** exponent for exponent in range(2, int(np.log10(max_queries)) + 1)]

    for row in (0, 1):
        axes[row, column].set_xscale('log')
        axes[row, column].set_yscale('log')
        axes[row, column].set_xlim(50, max_queries)
        axes[row, column].set_xticks(ticks)
        axes[row, column].minorticks_off()

    axes[2, column].set_xscale('log')
    axes[2, column].set_xlim(50, max_queries)
    axes[2, column].set_xticks(ticks)
    axes[2, column].set_ylim(0.90, 1.0)
    axes[2, column].minorticks_off()
    axes[2, column].set_xlabel('Number of Queries', fontsize=15)

axes[0, 0].set_ylim(9, 1.02e5)
axes[0, 1].set_ylim(0.98, 3.06e3)
axes[0, 2].set_ylim(9, 2e4)
axes[0, 3].set_ylim(9, 1.02e5)

axes[0, 0].set_ylabel('Cumulative Wall Time (s)', fontsize=15)
axes[1, 0].set_ylabel('Query Latency (ms)', fontsize=15)
axes[2, 0].set_ylabel('Recall', fontsize=15)

fig.legend(
    [legend_handles[name] for name in METHOD_ORDER],
    METHOD_ORDER,
    loc='upper center',
    ncol=len(METHOD_ORDER),
    fontsize=12,
    frameon=True,
)
fig.tight_layout()
fig.subplots_adjust(top=0.90)

output_path = FIGURE_DIR / 'main_acc_time.pdf'
fig.savefig(output_path, dpi=300, bbox_inches='tight')
print(f'Saved {output_path.relative_to(REPO_ROOT)}')
plt.show()
